In [ ]:
from IPython.display import clear_output
import numpy as np
import sys, random, time, math, os, copy, pickle
import pygame, argparse
import matplotlib.pyplot as plt
from IPython.display import clear_output
import cv2
np.set_printoptions(suppress=True)

def xy_state(x1, y1, x2, y2):
    d_x = np.sign(x1 - x2)
    d_y = np.sign(y1 - y2)
    n = 3 * d_x + d_y + 4
    if n > 4:
        n -= 1
    return n

def softmax(vect: list):
    vect = np.array(vect, dtype=float)
    vect = np.exp(vect - np.max(vect))
    return vect / np.sum(vect)

dist_names = ['left', 'up', 'right', 'down']
dict_vect = {'left': [-1, 0], 'up': [0, -1], 'right': [1, 0], 'down': [0, 1]}
moves_dict = {'left': 0, 'up': 1, 'right': 2, 'down': 3}

def get_is_wall(allowed_moves, last_move):
    result = np.zeros((5,))
    for current_move in allowed_moves:
        result[moves_dict[current_move]] = 1
    result[4] = int(last_move in allowed_moves)
    result = 1 - result
    return result

def visualize_detection(screenshot, pacman_pos, ghost_pos, pg_pos):
    """
    Визуализирует обнаруженные объекты на скриншоте.
    """
    vis_img = screenshot.copy()
    
    # Рисуем пакмана (зеленый круг)
    if pacman_pos[0] is not None and pacman_pos[1] is not None:
        cv2.circle(vis_img, (pacman_pos[0], pacman_pos[1]), 10, (0, 255, 0), 2)
        cv2.putText(vis_img, 'Pacman', (pacman_pos[0]-20, pacman_pos[1]-15), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    
    # Рисуем призрака (красный круг)
    if ghost_pos[0] is not None and ghost_pos[1] is not None:
        cv2.circle(vis_img, (ghost_pos[0], ghost_pos[1]), 10, (0, 0, 255), 2)
        cv2.putText(vis_img, 'Ghost', (ghost_pos[0]-15, ghost_pos[1]-15), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    # Рисуем ближайший пакгам (синий круг)
    if pg_pos[0] is not None and pg_pos[1] is not None:
        cv2.circle(vis_img, (pg_pos[0], pg_pos[1]), 5, (255, 0, 0), 2)
        cv2.putText(vis_img, 'PG', (pg_pos[0]-10, pg_pos[1]-10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    
    return vis_img

LEVELS = {}

LEVELS[1] = {
    'MODES': (
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 7 ],
        [ 'chase', 20 ],
        [ 'chase', 5 ],
        [ 'chase', 9999999 ]
    ),
    'red_dots_remaining': 20,
    'bonus': 'cherry'
}

FRUITS = {  "cherry": {  'id': 7, "score": 100 },
            "strawberry": {  'id': 8, "score": 300 },
            "orange": {  'id': 9, "score": 500 },
            "apple": {  'id': 10, "score": 700 },
            "melon": {  'id': 11, "score": 1000 },
            "galboss": {  'id': 12, "score": 2000 },
            "bell": {  'id': 13, "score": 3000 },
            "key": {  'id': 14, "score": 5000 }
}

SCATTER = { "red": (26,1) ,
            "blue": (26,29),
            "yellow": (1,29),
            "pink": (1,1)
}

JAIL = { "red": (12,14) ,
         "blue": (13,14),
         "yellow": (15,14),
         "pink": (14,14)
}


PACMAN_TIMERS = {
    "normal": 999999999,
    "chase": 6
}

PACMAN_POS = (13, 23)

FORBIDDEN_UP = [
    (12, 10),
    (15, 10),
    (12, 22),
    (15, 22)
]

# 28 (0-27) x 31 (0-30)
MAP_global = [
    [52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53],
    [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 52, 53,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
    [50,  1, 52, 48, 48, 48, 48, 48, 48, 48, 53,  1,  1, 50, 51,  1,  1, 52, 48, 48, 48, 48, 48, 48, 48, 53,  1, 51],
    [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 50, 51,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
    [50,  1, 54, 49, 49, 49, 49, 49, 49, 49, 55,  1,  1, 54, 55,  1,  1, 54, 49, 49, 49, 49, 49, 49, 49, 55,  1, 51],
    [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
    [50,  1, 52, 48, 48,  1, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,  1, 48, 48, 53,  1, 51],
    [50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51],
    [50,  1, 50,  1, 52, 48, 48, 48, 48,  1, 53,  1,  1, 52, 53,  1,  1, 52,  1, 48, 48, 48, 48, 53,  1, 51,  1, 51],
    [50,  1,  1,  1, 50,  1,  1,  1,  1,  1, 51,  1,  1, 50, 51,  1,  1, 50,  1,  1,  1,  1,  1, 51,  1,  1,  1, 51],
    [50,  1, 50,  1, 54,  1, 49, 49, 49, 49, 55,  1,  1, 54, 55,  1,  1, 54, 49, 49, 49, 49,  1, 55,  1, 51,  1, 51],
    [50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51],
    [50,  1, 54, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 55,  1, 51],
    [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
    [50,  1, 52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53,  1, 51],
    [50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51],
    [50,  1, 50,  1, 52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1, 52,  1, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48,  1, 48, 53,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1, 50,  1, 52, 48, 48, 48, 48, 48, 48, 48, 48, 48, 48, 53,  1, 51,  1, 51,  1, 51,  1, 51],
    [50,  1,  1,  1, 50,  1, 50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51,  1, 51,  1,  1,  1, 51],
    [50,  1, 50,  1, 50,  1, 50,  1, 54,  1, 49, 49, 49, 49, 49, 49, 49, 49,  1, 55,  1, 51,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1, 54, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 55,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51,  1, 51],
    [50,  1, 50,  1, 54,  1, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49,  1, 55,  1, 51,  1, 51],
    [50,  1, 50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51,  1, 51],
    [50,  1, 54, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 55,  1, 51],
    [50,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 51],
    [54, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 49, 55]
]

# define colors
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
class Pacman(pygame.sprite.Sprite):
    def __init__(self, my_game, x, y, input_Q):
        pygame.sprite.Sprite.__init__(self)
        self.game = my_game
        self.x = None
        self.y = None
        self.real_x = None
        self.real_y = None
        self.speed = None
        self.mode = None
        self.mode_changed = None
        self.direction = None
        self.image = None
        self.allowed_moves = None
        self.count_moves = None
        self.start_time = None
        self.miss_loops = None
        self.set_bonus = None
        # Параметры компьютерного зрения
        self.cv_debug = True  # Выводить отладочную информацию
        self.show_cv_window = True  # Показывать окно с визуализацией
        self.save_cv_frames = False  # Сохранять кадры на диск
        self.cv_update_frequency = 5  # Частота обновления CV (каждый N-й кадр)
        # Q-learning параметры
        self.Q = input_Q
        self.current_state = 0
        self.action = 0
        self.alpha = 0.1        # скорость обучения
        self.gamma = 0.9        # коэффициент дисконтирования
        self.epsilon = 0.3      # начальная вероятность случайного действия
        self.epsilon_decay = 0.995  # уменьшение epsilon
        self.epsilon_min = 0.01 # минимальный epsilon
        
        # Вспомогательные переменные
        self.old_dist = -1
        self.old_dist_ghost = -1
        self.last_moved = False
        self.reward = 0
        self.last_coordinates = None
        self.pacman_time = 0
        self.last_ate_time = 0
        self.last_nearest_pg = None
        self.last_move = 'left'
        
        self.reinit(x, y)

    def reinit(self, x, y):
        self.x = x
        self.y = y
        self.image = self.game.Pacman_pics['left'][1]
        self.rect = self.image.get_rect()
        self.rect.center = (self.x * 24 + 12, self.y * 24 + 12)
        self.real_x = self.x * 24 * 10
        self.real_y = self.y * 24 * 10
        self.speed = 55
        self.direction = "left"
        self.mode = "normal"
        self.allowed_moves = []
        self.count_moves = 0
        self.miss_loops = 0
        self.start_time = time.time()
        self.mode_changed = False
        self.radius = 3
        
        # Сбрасываем reward при переинициализации
        self.reward = 0
        self.old_dist = -1
        self.old_dist_ghost = -1
        self.pacman_time = 0
        self.last_ate_time = 0

    def get_q_table(self):
        return self.Q

    # Check what moves are allowed from this position
    def get_allowed_moves(self):
        """
        Check if pacman is allowed to move here
        """
        global MAP
        self.allowed_moves = []

        # check walls
        if MAP[self.y][self.x-1] < 16:
            self.allowed_moves.append("left")
        if (self.x+1 < 28 and MAP[self.y][self.x+1] < 16) or (self.x == 27 and self.direction == "right"):
            self.allowed_moves.append("right")
        if MAP[self.y - 1][self.x] < 16:
            self.allowed_moves.append("up")
        if MAP[self.y + 1][self.x] < 16:
            self.allowed_moves.append("down")

    def check_pacgums(self):
        global MAP
        chase = False
        if MAP[self.y][self.x] == 1:
            self.game.score += 10
            MAP[self.y][self.x] = 0
            self.game.pacgums -= 1
            self.miss_loops = 5
        return chase

    def change_mode(self):
        """
        Two modes for pacman: chase or normal
        """
        self.mode_changed = False
        current_time = time.time()

        # Enter chase mode
        if self.check_pacgums():
            self.mode = "chase"
            self.start_time = current_time
            self.mode_changed = True
            for ghost in self.game.Ghosts:
                if ghost.mode != "eaten":
                    ghost.change_mode("runaway")

        # rotate between modes based on timer
        else:
            mode_time = PACMAN_TIMERS[self.mode]
            if current_time - self.start_time > mode_time:
                if self.mode == "chase":
                    # Reinit ghosts in a row counter
                    self.game.ghosts_in_a_row = 0
                    self.mode = "normal"
                self.start_time = current_time
                self.mode_changed = True

        # if self.mode_changed:
        #     print("Pacman mode changed to", self.mode)

    def cv_analysis(self):
        """
        Анализ скриншота с помощью компьютерного зрения.
        Определяет позиции объектов и визуализирует результат.
        """
        try:
            # Создаем скриншот текущего экрана
            screenshot = pygame.surfarray.array3d(self.game.screen)
            screenshot = screenshot.transpose([1, 0, 2])  # Преобразуем для OpenCV
            
            # Конвертируем RGB в BGR (OpenCV использует BGR)
            screenshot_bgr = cv2.cvtColor(screenshot, cv2.COLOR_RGB2BGR)
            
            # Калибровка по нажатию клавиши 'c'
            keys = pygame.key.get_pressed()
            if keys[pygame.K_c]:
                self.calibrate_colors(screenshot_bgr)
                return
            
            # Определяем состояние с помощью компьютерного зрения
            cv_state, detected_objects = self.image_to_state_cv(screenshot_bgr)
            
            # Сравнение с внутренним состоянием
            if self.cv_debug:
                internal_state_str = f"Internal: {self.current_state}"
                cv_state_str = f"CV: {cv_state}"
                
                #if self.current_state != cv_state and cv_state != 0:
                #    print(f"STATE MISMATCH! {internal_state_str} vs {cv_state_str}")
                #else:
                #    print(f"States match: {internal_state_str}")
            
            # Создаем визуализацию
            vis_image = self.visualize_detection_cv(
                screenshot_bgr, 
                detected_objects['pacman'], 
                detected_objects['ghost'], 
                detected_objects['pg']
            )
            
            # Добавляем информацию о состояниях
            if self.cv_debug:
                cv2.putText(vis_image, f"Internal State: {self.current_state}", 
                        (10, vis_image.shape[0] - 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(vis_image, f"CV State: {cv_state}", 
                        (10, vis_image.shape[0] - 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Отображаем визуализацию в отдельном окне
            if self.show_cv_window:
                cv2.imshow('Computer Vision Analysis', vis_image)
                cv2.waitKey(1)  # Короткая задержка для отображения
                
        except Exception as e:
            if self.cv_debug:
                print(f"CV Analysis Error: {e}")

    def calibrate_colors(self, screenshot):
        """
        Функция для калибровки цветов в реальном времени.
        Позволяет кликнуть на объект и увидеть его цвет.
        """
        # Создаем копию для калибровки
        calib_img = screenshot.copy()
        
        def click_event(event, x, y, flags, param):
            if event == cv2.EVENT_LBUTTONDOWN:
                # Получаем цвет пикселя в разных цветовых пространствах
                bgr_color = screenshot[y, x]
                hsv_color = cv2.cvtColor(np.uint8([[bgr_color]]), cv2.COLOR_BGR2HSV)[0][0]
                
                print(f"Clicked at ({x}, {y})")
                print(f"BGR color: {bgr_color}")
                print(f"HSV color: {hsv_color}")
                print(f"Hex BGR: #{bgr_color[2]:02x}{bgr_color[1]:02x}{bgr_color[0]:02x}")
                print(f"Hex RGB: #{bgr_color[2]:02x}{bgr_color[1]:02x}{bgr_color[0]:02x}")
                print("-" * 50)
                
                # Рисуем круг на месте клика
                cv2.circle(calib_img, (x, y), 10, (0, 255, 0), 2)
                cv2.imshow('Calibration', calib_img)
        
        cv2.imshow('Calibration', calib_img)
        cv2.setMouseCallback('Calibration', click_event)
        cv2.waitKey(0)
        cv2.destroyWindow('Calibration')

    def image_to_state_cv(self, screenshot):
        """
        Определяет состояние из скриншота игры с новыми цветами текстур.
        Улучшенная версия с лучшей детекцией пакмана.
        """
        
        # ОПРЕДЕЛЯЕМ ГРАНИЦЫ ИГРОВОГО ПОЛЯ
        # Верхняя панель счета: примерно 32 пикселя
        # Нижняя панель с жизнями: примерно 40 пикселей
        top_panel_height = 32
        bottom_panel_height = 40
        
        # Высота и ширина скриншота
        screen_height, screen_width = screenshot.shape[:2]
        
        # Вычисляем границы игрового поля
        game_field_top = top_panel_height
        game_field_bottom = screen_height - bottom_panel_height
        game_field_left = 0
        game_field_right = screen_width
        
        # Обрезаем скриншот до игрового поля
        game_field = screenshot[game_field_top:game_field_bottom, 
                            game_field_left:game_field_right]
        
        if game_field.size == 0:
            # Если не удалось обрезать, используем весь скриншот
            game_field = screenshot
            game_field_top = 0
            game_field_bottom = screen_height
        
        # Создаем копию для отладки
        debug_img = game_field.copy() if hasattr(self, 'cv_debug') and self.cv_debug else None
        
        # Более точные диапазоны цветов для пакмана (основной цвет и дополнения)
        # Анализируем предоставленные цвета пакмана: 683b37, 774c47, 422820, e6d1cc, cab2ac, fff5f6, ad918b
        # Основной цвет пакмана: коричневато-розовые оттенки
        pacman_main_colors = [
            # Основные цвета пакмана (телесные/коричневые оттенки)
            np.array([0x37, 0x3b, 0x68]),   # 683b37 -> BGR: 55,59,104 (темный)
            np.array([0x47, 0x4c, 0x77]),   # 774c47 -> BGR: 71,76,119 (средний)
            np.array([0xcc, 0xd1, 0xe6]),   # e6d1cc -> BGR: 204,209,230 (светлый)
            np.array([0xac, 0xb2, 0xca]),   # cab2ac -> BGR: 172,178,202 (средне-светлый)
        ]
        
        # Дополнительные цвета для лучшего обнаружения
        pacman_secondary_colors = [
            np.array([0xf6, 0xf5, 0xff]),   # fff5f6 -> BGR: 246,245,255 (очень светлый)
            np.array([0x8b, 0x91, 0xad]),   # ad918b -> BGR: 139,145,173
            np.array([0x20, 0x28, 0x42]),   # 422820 -> BGR: 32,40,66 (очень темный)
        ]
        
        # Цвета призраков (сине-голубые оттенки)
        ghost_colors = [
            np.array([0xd4, 0xab, 0x62]),   # 62abd4 -> BGR: 212,171,98 (голубой)
            np.array([0xa1, 0x7e, 0x57]),   # 577ea1 -> BGR: 161,126,87 (темно-голубой)
            np.array([0xd0, 0xd5, 0xcd]),   # cdd5d0 -> BGR: 208,213,205 (серо-голубой)
            #np.array([0x1a, 0x13, 0x25]),   # 25131a -> BGR: 26,19,37 (очень темный)
        ]
        
        # Инициализируем переменные
        pacman_pos = (None, None)
        ghost_pos = (None, None)
        pg_pos = (None, None)
        
        # Преобразуем в HSV для лучшего разделения цветов
        hsv = cv2.cvtColor(game_field, cv2.COLOR_BGR2HSV)
        
        # --- УЛУЧШЕННЫЙ ПОИСК ПАКМАНА ---
        pacman_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        
        # Способ 1: Поиск по HSV диапазонам (лучше для телесных/коричневых оттенков)
        # Телесные/коричневые оттенки в HSV
        hsv_lower1 = np.array([0, 30, 100])    # Низкая насыщенность, средняя яркость
        hsv_upper1 = np.array([30, 150, 230])  # Красноватые/оранжевые оттенки
        
        # Сероватые/бежевые оттенки
        hsv_lower2 = np.array([0, 0, 150])     # Низкая насыщенность, высокая яркость
        hsv_upper2 = np.array([180, 50, 255])
        
        # Создаем маски по HSV
        hsv_mask1 = cv2.inRange(hsv, hsv_lower1, hsv_upper1)
        hsv_mask2 = cv2.inRange(hsv, hsv_lower2, hsv_upper2)
        pacman_hsv_mask = cv2.bitwise_or(hsv_mask1, hsv_mask2)
        
        # Способ 2: Поиск по BGR диапазонам (как было, но с улучшениями)
        for color in pacman_main_colors:
            # Широкий диапазон для основного цвета
            lower_bound = color - np.array([40, 40, 40])
            upper_bound = color + np.array([40, 40, 40])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            pacman_mask = cv2.bitwise_or(pacman_mask, color_mask)
        
        for color in pacman_secondary_colors:
            # Уже диапазон для вторичных цветов
            lower_bound = color - np.array([30, 30, 30])
            upper_bound = color + np.array([30, 30, 30])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            pacman_mask = cv2.bitwise_or(pacman_mask, color_mask)
        
        # Объединяем HSV и BGR маски
        pacman_mask = cv2.bitwise_or(pacman_mask, pacman_hsv_mask)
        
        if debug_img is not None:
            cv2.imshow('Pacman Mask Before', pacman_mask)
        
        # Морфологические операции для улучшения маски
        kernel = np.ones((5, 5), np.uint8)
        pacman_mask = cv2.morphologyEx(pacman_mask, cv2.MORPH_CLOSE, kernel)
        pacman_mask = cv2.morphologyEx(pacman_mask, cv2.MORPH_OPEN, kernel)
        
        # Удаляем слишком маленькие области (шум)
        contours, _ = cv2.findContours(pacman_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            # Фильтруем контуры по размеру
            filtered_contours = []
            for contour in contours:
                area = cv2.contourArea(contour)
                # Ожидаемый размер пакмана
                if 50 < area < 5000:  # Пакман должен быть среднего размера
                    filtered_contours.append(contour)
            
            if filtered_contours:
                # Берем самый большой подходящий контур
                largest_contour = max(filtered_contours, key=cv2.contourArea)
                
                # Более точное определение центра
                M = cv2.moments(largest_contour)
                if M['m00'] > 0:
                    pacman_x = int(M['m10'] / M['m00'])
                    pacman_y = int(M['m01'] / M['m00'])
                    
                    # Преобразуем координаты обратно в полный скриншот
                    pacman_x_full = pacman_x + game_field_left
                    pacman_y_full = pacman_y + game_field_top
                    pacman_pos = (pacman_x_full, pacman_y_full)
                    
                    # Дополнительная проверка: форма должна быть примерно круглой
                    perimeter = cv2.arcLength(largest_contour, True)
                    if perimeter > 0:
                        circularity = 4 * np.pi * M['m00'] / (perimeter * perimeter)
                        if circularity > 0.5:  # Достаточно круглая форма
                            pacman_pos = (pacman_x_full, pacman_y_full)
                            if debug_img is not None:
                                print(f"Pacman detected at {pacman_pos}, circularity: {circularity:.2f}")
        
        # Если не нашли пакмана обычным способом, пробуем найти по движению/позиции
        if pacman_pos[0] is None:
            # Попробуем найти самую яркую область в центре экрана (где обычно пакман)
            center_x, center_y = game_field.shape[1] // 2, game_field.shape[0] // 2
            search_radius = 100
            
            # Ищем области с высоким значением в HSV V канале (яркость)
            v_channel = hsv[:,:,2]
            search_area = v_channel[
                max(0, center_y-search_radius):min(game_field.shape[0], center_y+search_radius),
                max(0, center_x-search_radius):min(game_field.shape[1], center_x+search_radius)
            ]
            
            if search_area.size > 0:
                max_val = np.max(search_area)
                if max_val > 150:  # Достаточно яркая область
                    max_pos = np.unravel_index(np.argmax(search_area), search_area.shape)
                    pacman_x = center_x - search_radius + max_pos[1] + game_field_left
                    pacman_y = center_y - search_radius + max_pos[0] + game_field_top
                    pacman_pos = (pacman_x, pacman_y)
        
        # --- УЛУЧШЕННЫЙ ПОИСК ПРИЗРАКОВ ---
        ghost_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        
        # Поиск призраков по HSV (сине-голубые оттенки)
        ghost_hsv_lower = np.array([90, 50, 50])   # Синие оттенки
        ghost_hsv_upper = np.array([130, 255, 255])
        ghost_hsv_mask = cv2.inRange(hsv, ghost_hsv_lower, ghost_hsv_upper)
        
        # Поиск по BGR диапазонам
        for color in ghost_colors:
            lower_bound = color - np.array([30, 30, 30])
            upper_bound = color + np.array([30, 30, 30])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            ghost_mask = cv2.bitwise_or(ghost_mask, color_mask)
        
        # Объединяем маски
        ghost_mask = cv2.bitwise_or(ghost_mask, ghost_hsv_mask)
        
        # Убираем области, уже определенные как пакман
        if pacman_mask is not None:
            ghost_mask[pacman_mask > 0] = 0
        
        if debug_img is not None:
            cv2.imshow('Ghost Mask', ghost_mask)
        
        # Морфологические операции
        ghost_mask = cv2.morphologyEx(ghost_mask, cv2.MORPH_CLOSE, kernel)
        ghost_mask = cv2.morphologyEx(ghost_mask, cv2.MORPH_OPEN, kernel)
        
        # Находим контуры призраков
        ghost_contours, _ = cv2.findContours(ghost_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Находим ближайшего призрака к пакману
        nearest_ghost_pos = (None, None)
        
        if pacman_pos[0] is not None and ghost_contours:
            min_distance = float('inf')
            
            for contour in ghost_contours:
                area = cv2.contourArea(contour)
                # Призраки обычно среднего размера
                if 100 < area < 3000:
                    M = cv2.moments(contour)
                    if M['m00'] > 0:
                        ghost_x = int(M['m10'] / M['m00'])
                        ghost_y = int(M['m01'] / M['m00'])
                        
                        # Преобразуем координаты
                        ghost_x_full = ghost_x + game_field_left
                        ghost_y_full = ghost_y + game_field_top
                        
                        distance = np.sqrt((ghost_x_full - pacman_pos[0])**2 + (ghost_y_full - pacman_pos[1])**2)
                        
                        if distance < min_distance:
                            min_distance = distance
                            nearest_ghost_pos = (ghost_x_full, ghost_y_full)
        
        ghost_pos = nearest_ghost_pos
        
        # --- ПОИСК ПАКГАМОВ ---
        # Пакгамы обычно БЕЛЫЕ/СВЕТЛЫЕ маленькие точки
        # ИЛИ ярко-зеленые (если изменили цвет на 10ff00)
        pg_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        
        # Способ 1: Поиск ярко-зеленых точек (10ff00)
        # 10ff00 в HSV: зеленый оттенок
        green_lower = np.array([50, 100, 100])    # Нижняя граница зеленого
        green_upper = np.array([90, 255, 255])    # Верхняя граница зеленого
        green_mask = cv2.inRange(hsv, green_lower, green_upper)
        
        # Способ 2: Поиск белых/светлых точек
        white_lower = np.array([0, 0, 200])
        white_upper = np.array([180, 50, 255])
        white_mask = cv2.inRange(hsv, white_lower, white_upper)
        
        # Объединяем маски (зеленые и белые)
        pg_mask = cv2.bitwise_or(green_mask, white_mask)
        
        # Убираем области пакмана и призраков
        if pacman_mask is not None:
            pg_mask[pacman_mask > 0] = 0
        if ghost_mask is not None:
            pg_mask[ghost_mask > 0] = 0
        
        # Находим маленькие круглые объекты (пакгамы)
        pg_contours, _ = cv2.findContours(pg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        nearest_pg_pos = (None, None)
        if pacman_pos[0] is not None and pg_contours:
            min_distance = float('inf')
            
            for contour in pg_contours:
                area = cv2.contourArea(contour)
                # Пакгамы маленькие
                if 5 < area < 100:
                    M = cv2.moments(contour)
                    if M['m00'] > 0:
                        pg_x = int(M['m10'] / M['m00'])
                        pg_y = int(M['m01'] / M['m00'])
                        
                        # Преобразуем координаты
                        pg_x_full = pg_x + game_field_left
                        pg_y_full = pg_y + game_field_top
                        
                        distance = np.sqrt((pg_x_full - pacman_pos[0])**2 + (pg_y_full - pacman_pos[1])**2)
                        
                        if distance < min_distance:
                            min_distance = distance
                            nearest_pg_pos = (pg_x_full, pg_y_full)
        
        pg_pos = nearest_pg_pos
        
        # --- СОЗДАНИЕ СТЕЙТА И ОТЛАДКА ---
        current_state = 0
        allowed_moves_ = []
        
        if pacman_pos[0] is not None:
            # Если есть пакман, пытаемся определить доступные ходы
            wsz = 20
            
            # Простая проверка по яркости (используем игровое поле)
            gray = cv2.cvtColor(game_field, cv2.COLOR_BGR2GRAY)
            
            # Переводим координаты пакмана в систему координат игрового поля
            pacman_x_field = pacman_pos[0] - game_field_left
            pacman_y_field = pacman_pos[1] - game_field_top
            
            # Проверяем четыре направления
            directions = ['up', 'down', 'left', 'right']
            checks = [
                (pacman_y_field-wsz, pacman_y_field-5, pacman_x_field-5, pacman_x_field+5),  # up
                (pacman_y_field+5, pacman_y_field+wsz, pacman_x_field-5, pacman_x_field+5),  # down
                (pacman_y_field-5, pacman_y_field+5, pacman_x_field-wsz, pacman_x_field-5),  # left
                (pacman_y_field-5, pacman_y_field+5, pacman_x_field+5, pacman_x_field+wsz),  # right
            ]
            
            for direction, (y1, y2, x1, x2) in zip(directions, checks):
                if y1 >= 0 and y2 < gray.shape[0] and x1 >= 0 and x2 < gray.shape[1]:
                    area = gray[y1:y2, x1:x2]
                    if area.size > 0 and np.mean(area) > 100:  # Не слишком темно
                        allowed_moves_.append(direction)
            
            # Формируем состояние если все координаты есть
            if (ghost_pos[0] is not None and ghost_pos[1] is not None and
                pg_pos[0] is not None and pg_pos[1] is not None):
                
                try:
                    ghost_dir = xy_state(pacman_pos[0], pacman_pos[1], ghost_pos[0], ghost_pos[1])
                    pg_dir = xy_state(pacman_pos[0], pacman_pos[1], pg_pos[0], pg_pos[1])
                    
                    num_moves = len(allowed_moves_)
                    if num_moves < 2:
                        num_moves = 2
                    elif num_moves > 4:
                        num_moves = 4
                    
                    current_state = (ghost_dir * 8 + pg_dir) * 3 + (num_moves - 2)
                    
                    if debug_img is not None:
                        print(f"CV State: {current_state}, Pacman: {pacman_pos}, "
                            f"Ghost: {ghost_pos}, PG: {pg_pos}, Moves: {allowed_moves_}")
                        
                except Exception as e:
                    if debug_img is not None:
                        print(f"State calculation error: {e}")
        
        # Отладочная визуализация
        if debug_img is not None:
            # Показываем промежуточные маски
            cv2.imshow('Pacman Mask Final', pacman_mask)
            cv2.waitKey(1)
        
        # Возвращаем состояние и обнаруженные объекты
        detected_objects = {
            'pacman': pacman_pos,
            'ghost': ghost_pos,
            'pg': pg_pos,
            'allowed_moves': allowed_moves_
        }
        
        return current_state, detected_objects
    
    def visualize_detection_cv(self, screenshot, pacman_pos, ghost_pos, pg_pos):
        """
        Визуализирует обнаруженные объекты на скриншоте.
        """
        vis_img = screenshot.copy()
        
        # Рисуем пакмана (зеленый круг)
        if pacman_pos[0] is not None and pacman_pos[1] is not None:
            cv2.circle(vis_img, pacman_pos, 15, (0, 255, 0), 3)
            cv2.putText(vis_img, 'PACMAN', (pacman_pos[0]-25, pacman_pos[1]-20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        
        # Рисуем призрака (красный круг)
        if ghost_pos[0] is not None and ghost_pos[1] is not None:
            cv2.circle(vis_img, ghost_pos, 15, (0, 0, 255), 3)
            cv2.putText(vis_img, 'GHOST', (ghost_pos[0]-20, ghost_pos[1]-20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        
        # Рисуем ближайший пакгам (синий круг)
        if pg_pos[0] is not None and pg_pos[1] is not None:
            cv2.circle(vis_img, pg_pos, 8, (255, 200, 0), 3)
            cv2.putText(vis_img, 'PAC-GUM', (pg_pos[0]-25, pg_pos[1]-15), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 2)
        
        # Добавляем информационную панель
        info_y = 30
        cv2.putText(vis_img, f'Frame: {self.pacman_time}', (10, info_y), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(vis_img, f'Score: {self.game.score}', (10, info_y + 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(vis_img, f'Epsilon: {self.epsilon:.3f}', (10, info_y + 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        
        # Рисуем линии от пакмана к другим объектам
        if (pacman_pos[0] is not None and pacman_pos[1] is not None and
            ghost_pos[0] is not None and ghost_pos[1] is not None):
            cv2.line(vis_img, pacman_pos, ghost_pos, (0, 100, 255), 2)
            
        if (pacman_pos[0] is not None and pacman_pos[1] is not None and
            pg_pos[0] is not None and pg_pos[1] is not None):
            cv2.line(vis_img, pacman_pos, pg_pos, (255, 100, 0), 2)
        
        return vis_img

    # move pacman
    def update(self):
        global MAP
        current_speed = self.speed
        if self.miss_loops:
            current_speed -= self.miss_loops
            self.miss_loops = 0

        # если следующий шаг превышает следующую клетку
        if self.direction == 'left' and self.real_x-current_speed < (self.x-1)*24*10:
            next_speed = self.real_x - (self.x-1)*240
        elif self.direction == 'right' and self.real_x+current_speed > (self.x+1)*24*10:
            next_speed = (self.x+1)*240  - self.real_x
        elif self.direction == 'up' and self.real_y-current_speed < (self.y-1)*24*10:
            next_speed = self.real_y - (self.y-1)*240
        elif self.direction == 'down' and self.real_y+current_speed > (self.y+1)*24*10:
            next_speed = (self.y+1)*240 - self.real_y
        else:
            next_speed = current_speed

        next_loops = (next_speed, current_speed-next_speed)
        for speed in next_loops:
            if speed == 0:
                continue
                    
            # Выбираем направление только когда мы на координатах карты
            if (self.real_x / 10) % 24 == 0 and (self.real_y / 10) % 24 == 0:
                self.pacman_time += 1
                self.reward = 0
                self.x = int(self.real_x / 10 / 24)
                self.y = int(self.real_y / 10 / 24)
                self.change_mode()
                self.get_allowed_moves()
                
                # === 1. СОХРАНЯЕМ СТАРОЕ СОСТОЯНИЕ И ДЕЙСТВИЕ ===
                old_state = self.current_state
                old_action = self.action
                
                if self.game.score > self.game.old_score:
                    self.last_ate_time = self.pacman_time
                    self.reward += self.game.score - self.game.old_score

                # Находим ближайший пак-гам (внутренний метод)
                y_length = len(MAP)
                x_length = len(MAP[0])
                pacgum_list = []
                for y in range(y_length):
                    for x in range(x_length):
                        if MAP[y][x] == 1:
                            dist = ((x - self.x) ** 2 + (y - self.y) ** 2) ** 0.5
                            pacgum_list.append({'x': x, 'y': y, 'd': dist})
                
                if len(pacgum_list) > 0:
                    ind = np.argmin([pacgum_list[i]['d'] for i in range(len(pacgum_list))])
                    current_dist = pacgum_list[ind]['d']
                    pg_x = pacgum_list[ind]['x']
                    pg_y = pacgum_list[ind]['y']

                    # Награда за приближение к точке
                    if current_dist < self.old_dist and self.pacman_time - self.last_ate_time < 50:
                        self.reward += self.old_dist - current_dist
                    if current_dist > self.old_dist and self.pacman_time - self.last_ate_time > 50:
                        self.reward -= 0.5
                    
                    self.old_dist = pacgum_list[ind]['d']
                    
                    # Находим ближайшего призрака (внутренний метод)
                    ghost_list = []
                    for ghost in self.game.Ghosts:
                        dist = ((ghost.x - self.x) ** 2 + (ghost.y - self.y) ** 2) ** 0.5
                        ghost_list.append({'x': ghost.x, 'y': ghost.y, 'd': dist})
                    
                    if len(ghost_list) > 0:
                        ind = np.argmin([ghost_list[i]['d'] for i in range(len(ghost_list))])
                        ghost_x = ghost_list[ind]['x']
                        ghost_y = ghost_list[ind]['y']
                        
                        current_dist_ghost = ghost_list[ind]['d']
                        # Штраф за приближение призрака
                        if current_dist_ghost < self.old_dist_ghost:
                            self.reward -= 3 * (self.old_dist_ghost - current_dist_ghost)
                        
                        self.old_dist_ghost = ghost_list[ind]['d']
                        
                        # === 2. ВЫЧИСЛЯЕМ ТЕКУЩЕЕ СОСТОЯНИЕ ===
                        self.current_state = (xy_state(self.x, self.y, ghost_x, ghost_y) * 8 + 
                                            xy_state(self.x, self.y, pg_x, pg_y)) * 3 + len(self.allowed_moves) - 2
                        
                        # === КОМПЬЮТЕРНОЕ ЗРЕНИЕ: РАСПОЗНАВАНИЕ ИЗ ИЗОБРАЖЕНИЯ ===
                        # Делаем скриншот только каждые N кадров для производительности
                        if self.pacman_time % 5 == 0:  # Каждый 5-й кадр
                            self.cv_analysis()
                        
                        # === 3. ВЫБОР ДЕЙСТВИЯ (ε-greedy стратегия) ===
                        self.is_wall = get_is_wall(self.allowed_moves, self.last_move)
                        
                        # Получаем список допустимых действий
                        valid_actions = []
                        for i in range(5):  # 5 действий: left(0), up(1), right(2), down(3), stay(4)
                            if self.is_wall[i] == 0:  # 0 = можно двигаться
                                valid_actions.append(i)
                        
                        # ε-greedy выбор
                        if random.random() < self.epsilon and valid_actions:
                            # Случайное исследование
                            self.action = random.choice(valid_actions)
                        else:
                            # Эксплуатация: выбираем лучшее действие
                            q_values = self.Q[self.current_state, :].copy()
                            
                            # Устанавливаем очень низкие значения для запрещенных действий
                            q_values[self.is_wall == 1] = -1e6
                            
                            # Находим максимальное Q-значение
                            max_q = np.max(q_values)
                            
                            # Если несколько действий имеют одинаковое максимальное значение
                            best_actions = np.where(q_values == max_q)[0]
                            
                            # Выбираем случайное из лучших действий
                            if len(best_actions) > 0:
                                self.action = random.choice(best_actions.tolist())
                            else:
                                # Если все действия запрещены (не должно быть), выбираем stay
                                self.action = 4
                        
                        # === 4. ОБНОВЛЯЕМ НАПРАВЛЕНИЕ ДВИЖЕНИЯ ===
                        if self.action < 4:
                            self.direction = dist_names[self.action]
                        else:
                            self.direction = self.last_move
                        
                        # === 5. ОБНОВЛЕНИЕ Q-ТАБЛИЦЫ (Q-learning) ===
                        # Только если у нас есть предыдущее состояние и действие
                        if old_state is not None and old_action is not None:
                            # Получаем текущее Q-значение
                            current_q = self.Q[old_state, old_action]
                            
                            # Находим максимальное Q-значение для текущего (нового) состояния
                            next_q_values = self.Q[self.current_state, :].copy()
                            
                            # Учитываем стены для следующего состояния
                            next_is_wall = get_is_wall(self.allowed_moves, self.last_move)
                            next_q_values[next_is_wall == 1] = -1e6
                            
                            # Находим максимальное допустимое Q-значение
                            max_next_q = np.max(next_q_values) if len(next_q_values) > 0 else 0
                            
                            # Формула Q-learning: Q(s,a) = Q(s,a) + α * (reward + γ * maxQ(s',a') - Q(s,a))
                            new_q = current_q + self.alpha * (self.reward + self.gamma * max_next_q - current_q)
                            self.Q[old_state, old_action] = new_q
                            
                            # Уменьшаем epsilon (исследование)
                            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
                        
                        self.game.old_score = self.game.score
                
                # Управление с клавиатуры (для отладки)
                keys = pygame.key.get_pressed()
                if keys[pygame.K_LEFT]:
                    if MAP[self.y][self.x-1] < 16:
                        self.direction = "left"
                elif keys[pygame.K_RIGHT]:
                    if self.x+1 < 28 and MAP[self.y][self.x+1] < 16:
                        self.direction = "right"
                elif keys[pygame.K_UP]:
                    if MAP[self.y - 1][self.x] < 16:
                        self.direction = "up"
                elif keys[pygame.K_DOWN]:
                    if self.y + 1 < 30 and MAP[self.y + 1][self.x] < 16:
                        self.direction = "down"

            # === 6. ВЫПОЛНЕНИЕ ВЫБРАННОГО ДЕЙСТВИЯ (ПЕРЕМЕЩЕНИЕ) ===
            self.last_moved = True
            if self.direction == "left" and "left" in self.allowed_moves:
                self.last_move = 'left'
                self.real_x -= speed
                self.rect.x = round(self.real_x/10)
                # переход в правый туннель
                if self.rect.x <= -24:
                    self.rect.x = self.game.WIDTH-24
                    self.real_x = self.rect.x*10
            elif self.direction == "right" and "right" in self.allowed_moves:
                self.last_move = 'right'
                self.real_x += speed
                self.rect.x = round(self.real_x/10)
                # переход в левый туннель
                if self.rect.x >= self.game.WIDTH:
                    self.rect.x = -24
                    self.real_x = -240
            elif self.direction == "up" and "up" in self.allowed_moves:
                self.last_move = 'up'
                self.real_y -= speed
                self.rect.y = round(self.real_y/10)
            elif self.direction == "down" and "down" in self.allowed_moves:
                self.last_move = 'down'
                self.real_y += speed
                self.rect.y = round(self.real_y/10)
            else:
                self.last_moved = False
                self.last_move = ''
    
    # Обновляем позицию спрайта
        if self.last_moved:
            self.count_moves += 1
            self.rect.x = round(self.real_x/10) - 4
            self.rect.y = round(self.real_y/10) - 4
        
        # Обновляем анимацию
        if self.direction:
            if self.last_moved:
                self.image = self.game.Pacman_pics[self.direction][self.count_moves % 3 + 1]
            else:
                self.image = self.game.Pacman_pics[self.direction][2]

class Ghost(pygame.sprite.Sprite):
    """
    Defines a ghost
    """
    moves = ["left", "right", "up", "down"]
    opposite = ["right", "left", "down", "up"]

    def __init__(self, my_game, x, y, color, mode):
        pygame.sprite.Sprite.__init__(self)
        self.game = my_game
        self.x = None
        self.y = None
        self.real_x = None
        self.real_y = None
        self.color = color
        self.mode = None
        self.old_mode = None
        self.distances = None
        self.allowed_moves = None
        self.forbid_turnback = None
        self.in_tunnel = None
        self.direction = None
        self.speed = None
        self.count_moves = None
        self.blinking_tempo = None
        self.mode_changed = None
        self.start_time = None
        self.image = None
        self.target = None
        self.reinit(x, y, mode)

    def reinit(self, x, y, mode):
        self.x = x
        self.y = y
        self.mode = mode
        self.old_mode = ""
        self.distances = dict()
        self.allowed_moves = []
        self.forbid_turnback = True
        #self.direction = "left"
        self.direction = ''
        self.speed = 55
        self.blinking_tempo = 0
        self.in_tunnel = False
        self.radius = 3
        self.start_time = time.time()
        self.mode_changed = False
        if self.color in ('red', 'yellow', 'blue', 'pink'):
            self.image = self.game.Ghost_pics[self.color]['left'][1]
        else:
            self.image = self.game.Bonuses[self.color]
        self.rect = self.image.get_rect()
        self.rect.center = (self.x * 24 + 12, self.y * 24 + 12)
        self.real_x = self.x * 24 * 10
        self.real_y = self.y * 24 * 10

    # Pinky is pink
    def get_pinky_ambush_direction(self):
        """
        Pinky will go 4 tiles from pacman, depending its direction
        """
        if self.game.pacman.direction == "left":
            ambush_x = self.game.pacman.x - 4
            ambush_y = self.game.pacman.y
        elif self.game.pacman.direction == "right":
            ambush_x = self.game.pacman.x + 4
            ambush_y = self.game.pacman.y
        elif self.game.pacman.direction == "down":
            ambush_x = self.game.pacman.x
            ambush_y = self.game.pacman.y + 4
        else:
            ambush_x = self.game.pacman.x
            ambush_y = self.game.pacman.y - 4
        self.target = (ambush_x, ambush_y)

    # Inky is blue
    def get_inky_ambush_direction(self):
        """
        Inky will go to the opposite position 
        between Blinky (red) and two tiles from pacman
        """
        if self.game.pacman.direction == "left":
            ambush_x = self.game.pacman.x - 2
            ambush_y = self.game.pacman.y
        elif self.game.pacman.direction == "right":
            ambush_x = self.game.pacman.x + 2
            ambush_y = self.game.pacman.y
        elif self.game.pacman.direction == "down":
            ambush_x = self.game.pacman.x
            ambush_y = self.game.pacman.y + 2
        else:
            ambush_x = self.game.pacman.x - 2
            ambush_y = self.game.pacman.y - 2
        # Then, opposite direction from blinky (red) to this point
        ambush_x = ambush_x + ( self.game.red.x - ambush_x )
        ambush_y = ambush_y + ( self.game.red.y - ambush_y )
        self.target = (ambush_x, ambush_y)

    # Clyde is yellow
    def get_clyde_sneaking_direction(self):
        """
        Clyde (yellow) will chase pacman until it's too close.
        At less than 8 tiles, its direction becomes the scatter one
        """
        # Pythagore, of course
        dist_x = abs(self.x - self.game.pacman.x)
        dist_y = abs(self.y - self.game.pacman.y)
        distance = math.sqrt(dist_x**2 + dist_y**2)
        if distance >= 8:
            self.target = (self.game.pacman.x, self.game.pacman.y)
        else:
            self.target = (SCATTER[self.color][0], SCATTER[self.color][1])

    # In chase or runaway modes, we calculate distance between ghost and pacman
    def choose_direction(self):
        x = self.x
        y = self.y
        self.distances = dict()
        if self.forbid_turnback == False:
            self.forbid_turnback = True
        if self.mode == "scatter":
            self.target = (SCATTER[self.color][0], SCATTER[self.color][1])
        elif self.mode == "eaten":
            self.target = (JAIL[self.color][0], JAIL[self.color][1])
        elif self.mode == "chase":
            if self.color == "pink":
                self.get_pinky_ambush_direction()
            elif self.color == "blue":
                self.get_inky_ambush_direction()
            elif self.color == "yellow":
                self.get_clyde_sneaking_direction()
            elif self.color == "red":
                self.target = (self.game.pacman.x, self.game.pacman.y)
            else:
                self.target = (self.game.pacman.x, self.game.pacman.y)
        for direction in self.allowed_moves:
            x = self.x
            y = self.y
            if direction == "up":
                y = self.y - 1
            elif direction == "down":
                y = self.y + 1
            elif direction == "left":
                x = self.x - 1
            else:
                # right
                x = self.x + 1
            dist_x = abs(x - self.target[0])
            dist_y = abs(y - self.target[1])
            distance = math.sqrt(dist_x**2 + dist_y**2)
            self.distances[direction] = distance
        if self.mode in ("chase", "scatter", "eaten", "jail"):
            min_dist = 99999999999
        elif self.mode == "runaway":
            min_dist = -1
        for key, value in self.distances.items():
            # In chase mode : select the nearest direction
            if self.mode in ("chase", "scatter", "eaten" , "jail"):
                if value < min_dist:
                    min_dist = value
                    self.direction = key
            # In run away mode: select the farthest direction
            elif self.mode == "runaway":
                if value > min_dist:
                    min_dist = value
                    self.direction = key

    # Checks the free positions around the ghost
    def get_allowed_moves(self, x=None, y=None):
        global MAP
        """
        based on current position, list all possible directions
        """
        allowed_moves = []

        if not x:
            x = self.x
            y = self.y

        # check walls
        # 28 (0-27) x 31 (0-30)
        if MAP[y][x-1] < 16:
            allowed_moves.append("left")
        # Problems with right tunnel : a ghost can go there
        if (x+1 < 28 and MAP[y][x+1] < 16) or x+1 == 28:
            allowed_moves.append("right")
        # if not in jail mode , we can go outside
        if x<28 and (MAP[y - 1][x] < 16 or (self.mode != "jail" and MAP[y - 1][x] == 17)):
            allowed_moves.append("up")
        # You can only enter in jail in eaten mode
        if x<28 and (MAP[y + 1][x] < 16 or (self.mode == "eaten" and MAP[y + 1][x] == 17)):
            allowed_moves.append("down")

        # In some positions ghist isn't allowed to go up, except if eaten (to go directly in jail)
        if 'up' in allowed_moves and self.mode != 'eaten':
            for position in FORBIDDEN_UP:
                if x == position[0] and y-1 == position[1]:
                    allowed_moves.remove('up')
                    break

        # Remove opposition direction By default : no turn back
        if self.direction != '':
            reverse = self.opposite[self.moves.index(self.direction)]

            if self.forbid_turnback and reverse in allowed_moves:
                #if reverse in self.allowed_moves:
                allowed_moves.remove(reverse)

            # It should happen only if a row line
            if len(allowed_moves) == 0:
                allowed_moves.append(reverse)

        return(allowed_moves)

    # Check time spent in current mode then change it based on a timer
    def change_mode(self, new_mode=False):
        """
        Change the current move mode, based on a timer or a given value
        """
        self.mode_changed = False

        if new_mode:
            # start time of runaway is aways the same as pacman in chase
            self.old_mode = self.mode
            self.mode = new_mode
            self.mode_changed = True
        else:
            # Modes are managed by game loop
            # Runaway is sync with pacman current status
            if self.mode == 'runaway':
                if self.game.pacman.mode != 'chase':
                    self.mode = LEVELS[self.game.level]['MODES'][self.game.current_mode_idx][0]
                    self.get_speed()

            # Eaten and runaway are specific: don't change the mod during it
            if self.mode not in ('eaten', 'runaway'):
                self.mode = LEVELS[self.game.level]['MODES'][self.game.current_mode_idx][0]
                if self.old_mode != self.mode:
                    self.mode_changed = True
                    self.old_mode = self.mode

                if self.color == 'red' and self.game.pacgums <= LEVELS[self.game.level]['red_dots_remaining']:
                    self.mode = "chase"

        if self.mode_changed:
            self.get_speed()
            self.start_time = time.time()
            if self.mode not in ('scatter'):
                self.forbid_turnback = False
            # print(self.color, "mode changed to ", self.mode)

    def get_speed(self):
        # new speed ?
        if self.mode == "eaten":
            self.speed = 180
        elif self.mode == "runaway":
            self.speed = 20
        elif self.mode == "chase":
            self.speed = 55
        else:
            self.speed = 55

    # main function
    def update(self):
        global MAP
        # For blinking temporisation in Frightened mode
        self.blinking_tempo += 0.25

        # Check if we're in eaten and we are now on expected coordinates
        if self.mode == "eaten" and self.x == JAIL[self.color][0] and self.y == JAIL[self.color][1]:
            self.reinit(JAIL[self.color][0], JAIL[self.color][1], "jail")

        # change mode based on timer
        self.change_mode()

        # in tunnel divide speed by 2
        if MAP[self.y][self.x] == 15:
            self.in_tunnel = True
        else:
            self.in_tunnel = False

        current_speed = self.speed
        if self.in_tunnel:
            current_speed = int(self.speed/2)

        # if the next move exceeds the next case
        if self.direction == 'left' and self.real_x-current_speed < (self.x-1)*24*10:
            next_speed = self.real_x - (self.x-1)*240
        elif self.direction == 'right' and self.real_x+current_speed > (self.x+1)*24*10:
            next_speed = (self.x+1)*240  - self.real_x
        elif self.direction == 'up' and self.real_y-current_speed < (self.y-1)*24*10:
            next_speed = self.real_y - (self.y-1)*240
        elif self.direction == 'down' and self.real_y+current_speed > (self.y+1)*24*10:
            next_speed = (self.y+1)*240 - self.real_y
        else:
            next_speed = current_speed
        next_loops = (next_speed, current_speed-next_speed)
        #if self.color == 'red':
        #    print("next_loops=", next_loops)

        for speed in next_loops:
            if speed == 0:
                continue
            # Choose a direction only when we're on a MAP coordinates
            if (self.real_x /10) % 24 == 0 and (self.real_y/10) % 24 == 0:

                self.x = int(self.real_x / 10 / 24)
                self.y = int(self.real_y / 10 / 24)

                # What are the allowed moves ?
                self.allowed_moves = self.get_allowed_moves()
                self.choose_direction()

                # change mode alreay done, directino alreay set, we can forbid
                self.forbid_turnback = True

            #if self.color == "red":
            #    print(self.color, 'direction=', self.real_x, self.real_y, self.x, self.y,  self.direction, 'Original speed=', self.speed, 'real_speed', real_speed, 'loop_speed=',speed, 'loops values', next_loops)

            # Direction is set : move the ghost
            if self.direction == "left":
                self.real_x -= speed
                self.rect.x = round(self.real_x/10)
                # go to right tunnel 
                if self.rect.x <= -24 :
                    self.rect.x = self.game.WIDTH-24
                    self.real_x = self.rect.x*10
            elif self.direction == "right":
                self.real_x += speed
                self.rect.x = round(self.real_x/10)
                # go to left tunnel
                if self.rect.x >= self.game.WIDTH:
                    self.rect.x = -24
                    self.real_x = -240
            elif self.direction == "up":
                self.real_y -= speed
                self.rect.y = round(self.real_y/10)
            elif self.direction == "down":
                self.real_y += speed
                self.rect.y = round(self.real_y/10)

        self.rect.x = round(self.real_x/10) - 4
        self.rect.y = round(self.real_y/10) - 4

        # Ghosts bitmaps depending of the status
        if self.mode == "runaway":
            current_time = time.time()
            # The last 3 seconds : blink
            if PACMAN_TIMERS['chase'] - (current_time - self.start_time) < 3:
                self.image = self.game.Frightened_ghost_blinking[int(self.blinking_tempo) % 4 + 1]
            else:
                self.image = self.game.Frightened_ghost[int(self.blinking_tempo) % 2 + 1]
        elif self.mode == "eaten":
            self.image = self.game.Ghost_eyes[self.direction]
        else:
            self.image = self.game.Ghost_pics[self.color][self.direction][int(self.blinking_tempo) % 2 + 1]

class Game:
    """
    Main class that manages the full game
    """
    def __init__(self, input_Q, max_loops):
        global MAP
        self.theme = "default"
        self.dymmy = None
        self.Pacman_pics = None
        self.Dead_pacman = None
        self.Ghost_pics = None
        self.Ghost_eyes = None
        self.Scores = None
        self.Frightened_ghost = None
        self.Frightened_ghost_blinking = None
        self.Walls = None
        self.Bonuses = None
        self.Ready = None
        self.ghosts_in_a_row = 0
        self.win = False
        self.trunc = False
        self.max_count_loops = max_loops

        # ghosts are set using setattr but pylint is yelling...
        self.pink = None
        self.red = None
        self.blue = None
        self.yellow = None
        self.munch = None
        self.WIDTH = len(MAP[0])*24
        self.HEIGHT = len(MAP)*24
        self.FULL_WIDTH = self.WIDTH
        self.FULL_HEIGHT = self.HEIGHT + 72
        self.lifes = 3
        self.pacgums = self.count_pacgums()
        self.old_score = 0
        self.score = 0
        self.scale = 1
        self.FPS = 30
        self.count_loops = 0
        self.level = 0
        self.start_mode_timer = None
        self.current_mode_idx = 0
        self.current_mode = None
        pygame.init()
        self.parse_args()
        display_infos = pygame.display.Info()
        y_resolution = display_infos.current_h
        if y_resolution < 800:
            self.scale = 0.75
        self.screen = pygame.display.set_mode((int(self.FULL_WIDTH * self.scale), int(self.FULL_HEIGHT * self.scale)))
        self.fake_screen = pygame.Surface((self.FULL_WIDTH, self.FULL_HEIGHT))
        self.surface = pygame.Surface((self.WIDTH, self.HEIGHT))
        self.top = pygame.Surface((self.WIDTH, 32))
        self.bottom = pygame.Surface((self.WIDTH, 40))
        self.load_bitmaps()
        self.all_sprites = pygame.sprite.Group()
        self.all_ghosts = pygame.sprite.Group()
        self.bonus = None
        self.red = Ghost(self, 13, 11, 'red', "scatter")
        for color in ("blue", "yellow", "pink"):
            setattr(self, color, Ghost(self, JAIL[color][0], JAIL[color][1], color, "jail") )
        self.Ghosts = []
        self.Ghosts.append(self.pink)
        # self.Ghosts.append(self.red)
        # self.Ghosts.append(self.yellow)
        # self.Ghosts.append(self.blue)
        self.pacman = Pacman(self, PACMAN_POS[0], PACMAN_POS[1], input_Q)
        self.all_sprites.add(self.pacman)
        self.all_sprites.add(self.Ghosts)
        self.all_ghosts.add(self.Ghosts)

    def parse_args(self):
        parser = argparse.ArgumentParser()
        parser.add_argument('-t', '--theme', help='Theme name', default='default')
        args, unknown = parser.parse_known_args()
        # args = parser.parse_args()
        if not os.path.isdir(os.path.join(os.path.dirname(os.path.abspath('')), 'themes/'+args.theme)):
            self.theme = 'default'
        else:
            self.theme = args.theme

    def load_bitmaps(self):
        global MAP
        game_folder = os.getcwd()
        img_folder = os.path.join(game_folder, 'themes\\'+self.theme+'\\img')
        self.Frightened_ghost = dict()
        self.Frightened_ghost[1] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_1.png')).convert(),(32,32))
        self.Frightened_ghost[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_2.png')).convert(),(32,32))
        self.Frightened_ghost[1].set_colorkey(BLACK)
        self.Frightened_ghost[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking = dict()
        self.Frightened_ghost_blinking[1] = self.Frightened_ghost[1]
        self.Frightened_ghost_blinking[2] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_4.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[3] = self.Frightened_ghost[2]
        self.Frightened_ghost_blinking[4] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'frightened_ghost_3.png')).convert(),(32,32))
        self.Frightened_ghost_blinking[2].set_colorkey(BLACK)
        self.Frightened_ghost_blinking[4].set_colorkey(BLACK)
        self.Ghost_pics = dict()
        for color in ('red', 'yellow', 'blue', 'pink'):
            self.Ghost_pics[color] = dict()
            for direction in ('left', 'right', 'up', 'down'):
                self.Ghost_pics[color][direction] = dict()
                for i in range(1, 3):
                    self.Ghost_pics[color][direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, color+'_'+direction+'_ghost_'+str(i)+'.png')).convert(),(32,32))
                    self.Ghost_pics[color][direction][i].set_colorkey(BLACK)
        self.Ghost_eyes = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Ghost_eyes[direction] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'ghost_eyes_'+direction+'.png')).convert(),(32,32))
            self.Ghost_eyes[direction].set_colorkey(BLACK)
        self.Pacman_pics = dict()
        for direction in ('left', 'right', 'up', 'down'):
            self.Pacman_pics[direction] = dict()
            for i in range(1, 4):
                self.Pacman_pics[direction][i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_'+direction+'_'+str(i)+'.png')).convert(),(32,32))
                self.Pacman_pics[direction][i].set_colorkey(BLACK)
        self.Dead_pacman = dict()
        for i in range(1, 11):
            self.Dead_pacman[i] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, 'pacman_dead_'+str(i)+'.png')).convert(),(32,32))
            self.Dead_pacman[i].set_colorkey(BLACK)
        self.Scores = dict()
        for score in (100, 200, 300, 400, 500, 700, 800, 1000, 1600, 2000, 3000, 5000):
            self.Scores[score] = pygame.transform.smoothscale(pygame.image.load(os.path.join(img_folder, str(score)+'.png')).convert(),(32,32))
            self.Scores[score].set_colorkey(BLACK)
        self.Walls = dict()
        for l in MAP:
            for c in l:
                if c not in self.Walls:
                    png = os.path.join(img_folder, str(c) + ".png")
                    if os.path.exists(png):
                        self.Walls[c] = pygame.image.load(png).convert()
        for bonus in range(7,15):
            self.Walls[bonus] = pygame.image.load(os.path.join(img_folder, str(bonus)+'.png')).convert()
            self.Walls[bonus].set_colorkey(BLACK)
        self.Ready = pygame.image.load(os.path.join(img_folder, 'ready.png')).convert()
        self.Ready.set_colorkey(BLACK)

    def display_board_game(self):
        self.clear_all_surfaces()
        self.display_score()
        self.display_map(self.surface)
        self.all_sprites.draw(self.surface)
        self.display_lifes(self.bottom)
        self.blit_all_surfaces()

    def display_text(self, my_surface, my_text, pos_x, pos_y, color=WHITE):
        font = pygame.font.Font('RetroGaming.ttf', 18)
        text = font.render(my_text, True, color)
        my_surface.blit(text, (pos_x, pos_y))

    def display_score(self):
        self.display_text(self.top, "Player 1     Score: {}".format(self.score), 24, 6)

    def display_lifes(self, my_surface):
        for life in range(0, self.lifes - 1):
            my_surface.blit(self.Pacman_pics['right'][2], (life*32+24, 4))

    def start_game(self):
        i = 0
        while i<4:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            i += 1 
            self.clear_all_surfaces()
            self.display_map(self.surface)
            self.display_score()
            self.display_map(self.surface)
            self.all_sprites.draw(self.surface)
            self.display_lifes(self.bottom)
            self.surface.blit(self.Ready, (11*24, 17*24))
            self.blit_all_surfaces()
            pygame.display.flip()
            time.sleep(0.001)

    def loose_life(self):
        global MAP
        self.lifes -= 1
        # remove bonus
        MAP[17][13] = 0
        i = 1
        self.pacman.Q[self.pacman.current_state, self.pacman.action] = 0.99 * self.pacman.Q[self.pacman.current_state, self.pacman.action] - 0.01 * 1000
        while i < 15:
            # evaluate the pygame event
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    break
            self.clear_all_surfaces()
            self.display_map(self.surface)
            if i<11:
                self.surface.blit(self.Dead_pacman[i], (self.pacman.rect.x, self.pacman.rect.y))
            else:
                if i%2:
                    self.surface.blit(self.Dead_pacman[10], (self.pacman.rect.x, self.pacman.rect.y))
            self.display_lifes(self.bottom)
            self.blit_all_surfaces()
            pygame.display.flip()
            i += 1
            time.sleep(0.001)
        pygame.display.update()
        self.pacman.reinit(PACMAN_POS[0], PACMAN_POS[1])
        for ghost in self.Ghosts:
            ghost.reinit(JAIL[ghost.color][0], JAIL[ghost.color][1], "jail")
        if self.lifes > 0:
            self.start_game()
        self.current_mode_idx = 0
        self.start_mode_timer = time.time()

    def clear_all_surfaces(self):
        self.fake_screen.fill(BLACK)
        self.surface.fill(BLACK)
        self.top.fill(BLACK)
        self.bottom.fill(BLACK)

    def blit_all_surfaces(self):
        self.fake_screen.blit(self.top, (0, 0))
        self.fake_screen.blit(self.surface, (0, 32))
        self.fake_screen.blit(self.bottom, (0, self.HEIGHT+32))
        self.screen.blit(self.scale_output(self.fake_screen, self.scale), (0, 0))

    def scale_output(self, my_surface, my_scale):
        if my_scale != 1:
            frame = pygame.transform.scale(my_surface, (int(self.FULL_WIDTH * my_scale), int(self.FULL_HEIGHT * my_scale)))
        else:
            frame = my_surface
        return frame

    def count_pacgums(self):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        self.pacgums = 0
        for y in range(y_length):
            for x in range(x_length):
                if MAP[y][x] in (1, 2):
                    self.pacgums += 1
        return self.pacgums

    def display_map(self, my_surface):
        global MAP
        y_length = len(MAP)
        x_length = len(MAP[0])
        for y in range(y_length):
            for x in range(x_length):
                c = MAP[y][x]
                if c in self.Walls:
                    if c != 2:
                        my_surface.blit(self.Walls[c], (x*24, y*24))
                    else:
                        if self.count_loops%8 in range(0,3):
                            my_surface.blit(self.Walls[c], (x*24, y*24))

    def collided(self):
        hit_list = pygame.sprite.spritecollide(self.pacman, self.all_ghosts, False, pygame.sprite.collide_circle)
        return hit_list

    def play(self):
        self.ghosts_in_a_row = 0
        lives_gained = 0
        clock = pygame.time.Clock()
        running = True
        while running and self.level <= 1:
            self.count_loops = 0
            self.start_game()
            self.level += 1
            self.current_mode_idx = 0
            self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
            self.start_mode_timer = time.time()
            while running:
                clock.tick(self.FPS)
                self.count_loops += 1
                
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        running = False
                current_time = time.time()
                if current_time - self.start_mode_timer > LEVELS[self.level]['MODES'][self.current_mode_idx][1]:
                    self.start_mode_timer = current_time
                    self.current_mode_idx += 1
                    self.current_mode = LEVELS[self.level]['MODES'][self.current_mode_idx][0]
                self.pacman.update()
                self.all_ghosts.update()




                
                if self.count_loops % 2 == 0:




                    
                    # print(self.score)
                    self.display_board_game()
                pygame.display.flip()

                # Collision test
                hit_list = self.collided()
                if hit_list:
                    if self.pacman.mode == "chase":
                        for ghost in hit_list:
                            if ghost.mode == "runaway":
                                ghost.change_mode("eaten")
                                self.ghosts_in_a_row += 1
                                ghost_score = 200*math.pow(2, self.ghosts_in_a_row-1)
                                self.score += ghost_score
                            elif ghost.mode != "eaten":
                                self.loose_life()
                    else:
                        for ghost in hit_list:
                            if ghost.mode not in ("eaten", "runaway"):
                                self.loose_life()
                                break
                if self.pacgums == 0:
                    running = False
                    self.win = True
                if self.lifes == 0:
                    running = False
                if self.count_loops > self.max_count_loops:
                    running = False
                    self.trunc = True
        return (self.pacman.get_q_table(), self.score, self.count_loops, self.win, self.trunc)

# Root code
def main():
    global MAP
    
    # Инициализация окна OpenCV для визуализации
    cv2.namedWindow('Computer Vision Analysis', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Computer Vision Analysis', 600, 800)

    # Инициализация Q-таблицы с нуля
    # Количество состояний: (8 направлений к призраку) * (8 направлений к точке) * (3 варианта allowed_moves)
    num_states = 8 * 8 * 3  # 192 состояния
    input_q_table = np.zeros((num_states, 5))
    
    # Параметры обучения
    num_episodes = 1
    scores = []
    
    for episode in range(num_episodes):
        MAP = copy.deepcopy(MAP_global)
        game = Game(input_q_table, 5000)
        
        # Обновляем Q-таблицу после игры
        input_q_table, score_, iteration_, win_, trunc_ = game.play()
        scores.append(score_)
        
        # Выводим прогресс
        if episode % 10 == 0:
            avg_score = np.mean(scores[-10:]) if len(scores) >= 10 else np.mean(scores)
            print(f"Episode {episode}, Score: {score_}, Avg (last 10): {avg_score:.1f}")
    
    pygame.quit()
    
    # Сохраняем обученную таблицу
    with open("q_table_trained.pkl", "wb") as file:
        pickle.dump(input_q_table, file)
    
    # Выводим статистику
    print(f"\nTraining complete!")
    print(f"Best score: {max(scores)}")
    print(f"Average score: {np.mean(scores):.1f}")
        
if __name__ == "__main__":
    main()

c:\Users\byran\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.11)
Hello from the pygame community. https://www.pygame.org/contribute.html
Pacman detected at (381, 596), circularity: 0.76
CV State: 54, Pacman: (381, 596), Ghost: (414, 408), PG: (419, 595), Moves: ['up']
Pacman detected at (338, 596), circularity: 0.69
CV State: 69, Pacman: (338, 596), Ghost: (535, 408), PG: (299, 595), Moves: ['down', 'right']
Pacman detected at (315, 596), circularity: 0.69
CV State: 69, Pacman: (315, 596), Ghost: (583, 472), PG: (275, 595), Moves: ['down', 'right']
Pacman detected at (307, 596), circularity: 0.76
CV State: 69, Pacman: (307, 596), Ghost: (583, 593), PG: (275, 595), Moves: ['up']
Pacman detected at (413, 596), circularity: 0.69
CV State: 6, Pacman: (413, 596), Ghost: (565, 696), PG: (443, 595), Moves: ['down', 'right']
Pacman detected at (305, 596), circularity: 0.68
CV State: 21, Pacman: (305, 596), Ghost: (444, 696), PG: (275, 595), Moves: ['up', 'down']
Pacman detected at (233, 596), circularity: 0.76
CV St

In [ ]:
with open("q_table_no_ghost.pkl", "rb") as file:
    input_q_table = pickle.load(file)
input_q_table

array([[  7.58099449, -11.8015208 , -12.14822918,  -7.58979382,
        -10.86064008],
       [ -3.97053283,  -7.74099034,  -8.29614752,   4.44075706,
         -9.0166283 ],
       [  0.7547511 ,  -1.32278699,  -1.01520723,  -1.41465594,
         -1.26168467],
       [ 20.17860042,  -4.48479072,  44.73291397,  -2.77787166,
          6.47872688],
       [ -1.07219793,  -2.11659536,  40.37245418,  -1.1217943 ,
          0.92179371],
       [  0.45680154,  -0.38364188,   4.27411799,   0.62935997,
         29.00096353],
       [ -2.16192021,  -4.04771658,  -1.20637341,  -5.44354841,
          7.7248446 ],
       [  0.61472721,  -2.59664239,  -2.96062169,  -2.79265542,
          1.91855798],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ],
       [ -6.46014056,  -6.8311242 ,  -7.9149573 ,  -8.35836552,
         -5.8788975 ],
       [ -8.51551991,  -5.81611182,  -5.2220926 ,  -6.58805601,
         -4.97787676],
       [ -2.30455482,  -2.24693274,  -2.430